# 03 - Veri Ön İşleme ve Ortak Pipeline

**Sorumlu:** Kişi 3

**Ödevdeki karşılığı:** 2. Veri Ön İşleme (D1, D2, D3, D6, D7 Görevleri)

**Amaç:** Bu not defterinde, Bank Marketing veri seti üzerinde eksik ve `unknown` değerlerin yönetimi, sayısal değişkenlerin ölçeklendirilmesi, kategorik değişkenlerin kodlanması, ortak `ColumnTransformer` ve `Pipeline` mimarisinin inşası ile veri sızıntısını önleyecek %80-%20 tabakalı (stratified) veri ayrımı gerçekleştirilmektedir.

## 1. Çalışma Ortamı ve Kütüphanelerin Yüklenmesi
> **Açıklama:** Bu hücrede veri analizi kütüphaneleri (pandas, numpy) ile tüm modellerin ortak kullandığı ön işleme ve veri bölme modülleri (`src/ortak.py`, `src/preprocessing.py`) içe aktarılır.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Çalışma dizinini kontrol et ve proje kök dizinine sabitle
mevcut_dizin = Path.cwd()
if mevcut_dizin.name == "notebooks":
    os.chdir(mevcut_dizin.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Ortak veri yükleme, bölme, ön işleme ve doğrulama fonksiyonları
from src.ortak import veri_yukle, veriyi_bol, on_isleyici, ortak_cv, RANDOM_STATE
from src.preprocessing import (
    numeric_features, mode_features, unknown_features, other_cat_features
)

print("Aktif Çalışma Dizini:", Path.cwd())
print("Ortak modüller (src.ortak ve src.preprocessing) başarıyla yüklendi!")

Aktif Çalışma Dizini: c:\Users\nusayba\OneDrive\Masaüstü\makineprojesi\ml-proje
Ortak modüller (src.ortak ve src.preprocessing) başarıyla yüklendi!


## 2. Veri Setinin Ortak Fonksiyonla Okunması ve Boyut Kontrolü
> **Açıklama:** Bank Marketing verisi `src.ortak.veri_yukle()` ortak fonksiyonu aracılığıyla noktalı virgül (`;`) ayırıcısıyla okunur. Veri boyutu, satır ve sütun sayıları kontrol edilir.

In [2]:
# Veri setini ortak veri_yukle() fonksiyonu ile güvenli şekilde okuyalım
df = veri_yukle()

print("VERİ BOYUTU (veri_yukle sonrası):")
print(f"Satır Sayısı: {df.shape[0]}, Sütun Sayısı: {df.shape[1]}")
print("\nİlk 5 Satır:")
display(df.head(5))

VERİ BOYUTU (veri_yukle sonrası):
Satır Sayısı: 41188, Sütun Sayısı: 22

İlk 5 Satır:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,daha_once_aranmis
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0


**Yorum: Veri Seti Yapısı ve Ortak Yükleme Adımları:**
Bank Marketing ham veri seti 41.188 satır ve 21 sütundan oluşmaktadır. Ortak `veri_yukle()` fonksiyonu bu ham veriyi `;` ayırıcısıyla okur ve tüm modellerde tutarlılık sağlamak için aşağıdaki deterministik (veri öğrenmeyen) hazırlıkları uygular:
- Düşük oranlı `unknown` değerlerini (`job`, `marital`, `education`, `housing`, `loan`) `NaN` olarak işaretler; `default` sütunundaki `unknown` kategorisini korur.
- `pdays = 999` özel kodunu çözümleyerek `daha_once_aranmis` (0/1) ikili değişkenini türetir ve `pdays = 999` değerini `-1` yapar (bu nedenle sütun sayısı 21'den 22'ye çıkar).
- Hedef sütun olan `y` değişkenini `yes -> 1`, `no -> 0` şeklinde ikili formata çevirir.

## D1. Eksik, Bilinmeyen (Unknown) ve Tutarsız Değerlerin Analizi
> **Açıklama:** Veri setinde standart `NaN` değerleri ile gizli eksik değer olan `unknown` kayıtları tespit edilir. Sayısal sütunlarda ise `pdays = 999` özel kodu ve IQR tabanlı aykırı değerler incelenir.

In [3]:
# 1. Düşük oranlı sütunlarda 'unknown' -> NaN dönüşümünün kontrolü
nan_sayilari = df.isnull().sum()

# 2. Ayrı kategori olarak korunan 'unknown' değerlerinin tespiti (default sütunu)
unknown_sayilari = (df == "unknown").sum()

eksik_tablosu = pd.DataFrame({
    "NaN Sayısı (Unknown'dan Dönüşen)": nan_sayilari,
    "NaN Oranı (%)": (nan_sayilari / len(df) * 100).round(2),
    "Korunan Unknown Sayısı": unknown_sayilari,
    "Korunan Unknown Oranı (%)": (unknown_sayilari / len(df) * 100).round(2)
})

# Sadece eksiklik veya unknown içeren sütunları filtreleyelim
eksik_tablosu = eksik_tablosu[
    (eksik_tablosu["NaN Sayısı (Unknown'dan Dönüşen)"] > 0)
    | (eksik_tablosu["Korunan Unknown Sayısı"] > 0)
]
print("Eksik (NaN) ve Korunan Unknown Değer Dağılımı:")
display(eksik_tablosu)

Eksik (NaN) ve Korunan Unknown Değer Dağılımı:


,NaN Sayısı (Unknown'dan Dönüşen),NaN Oranı (%),Korunan Unknown Sayısı,Korunan Unknown Oranı (%)
job,330,0.80,0,0.00
marital,80,0.19,0,0.00
education,1731,4.20,0,0.00
default,0,0.00,8597,20.87
housing,990,2.40,0,0.00
loan,990,2.40,0,0.00


**Yorum: Eksik ve Unknown Değer Yönetim Stratejimiz:**
1. Ham veri setinde standart `NaN` bulunmamakta, ancak 6 farklı kategorik değişkende `unknown` şeklinde kodlanmış gizli eksik veriler yer almaktadır.
2. **`job` (330 adet - %0.80), `marital` (80 adet - %0.19), `education` (1.731 adet - %4.20), `housing` (990 adet - %2.40), `loan` (990 adet - %2.40):** Bu sütunlardaki `unknown` oranı düşüktür (%5'in altındadır). Satırları silmek veri kaybına (özellikle zaten az olan pozitif `yes` sınıfının kaybına) yol açacağı için, `prepare_unknown_values()` ile `NaN`'a dönüştürülmüş olup ortak `ColumnTransformer` içinde yalnızca eğitim verisinden öğrenilen en sık değerle (**mode / most_frequent**) doldurulacaktır.
3. **`default` (8.597 adet - %20.87):** Kredi temerrüdü sütununda `unknown` oranı %20.87'dir. Veri setinde temerrüde düşmüş (`yes`) müşteri sayısı yalnızca 3 adettir, `no` diyenler 32.588 kişidir. `unknown` değerini mode (`no`) ile doldurmak temerrüt risk belirsizliğini tamamen ortadan kaldırır. Müşterinin temerrüt durumunun bilinmemesi kendi başına ayırt edici bir bilgi taşıdığı için bu sütundaki `unknown` değeri **ayrı bir kategori olarak korunmuştur**.

In [4]:
# pdays özel kodu çözümlemesinin kontrolü ve sayısal sütunlarda IQR tabanlı aykırı değer incelemesi
aranmamis_sayisi = int((df["daha_once_aranmis"] == 0).sum())
aranmamis_oran = (df["daha_once_aranmis"] == 0).mean() * 100
print(f"Daha önce aranmamış müşteri (ham pdays = 999 -> pdays = -1) sayısı: {aranmamis_sayisi} (%{aranmamis_oran:.2f})")
print(f"Daha önce aranmış müşteri (daha_once_aranmis = 1) sayısı           : {len(df) - aranmamis_sayisi} (%{100 - aranmamis_oran:.2f})")

sayisal_sutunlar = [col for col in df.select_dtypes(include=["number"]).columns if col != "y"]
aykiri_raporu = []

for col in sayisal_sutunlar:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    alt_sinir = Q1 - 1.5 * IQR
    ust_sinir = Q3 + 1.5 * IQR
    aykiri_sayisi = int(((df[col] < alt_sinir) | (df[col] > ust_sinir)).sum())
    aykiri_raporu.append({
        "Sütun": col,
        "Min": df[col].min(),
        "Medyan": df[col].median(),
        "Maksimum": df[col].max(),
        "Aykırı Sayısı": aykiri_sayisi,
        "Aykırı Oranı (%)": round((aykiri_sayisi / len(df)) * 100, 2)
    })

display(pd.DataFrame(aykiri_raporu))

Daha önce aranmamış müşteri (ham pdays = 999 -> pdays = -1) sayısı: 39673 (%96.32)
Daha önce aranmış müşteri (daha_once_aranmis = 1) sayısı           : 1515 (%3.68)


,Sütun,Min,Medyan,Maksimum,Aykırı Sayısı,Aykırı Oranı (%)
0,age,17.000,38.000,98.000,469,1.14
1,duration,0.000,180.000,4918.000,2963,7.19
2,campaign,1.000,2.000,56.000,2406,5.84
3,pdays,-1.000,-1.000,27.000,1515,3.68
4,previous,0.000,0.000,7.000,5625,13.66
5,emp.var.rate,-3.400,1.100,1.400,0,0.00
6,cons.price.idx,92.201,93.749,94.767,0,0.00
7,cons.conf.idx,-50.800,-41.800,-26.900,447,1.09
8,euribor3m,0.634,4.857,5.045,0,0.00
9,nr.employed,4963.600,5191.000,5228.100,0,0.00


**Yorum: Sayısal Sütunlar, pdays Özel Kodu ve Aykırı Değerler:**
1. **`pdays`:** Ham veri setinin %96.32'sinde (39.673 kayıt) `pdays = 999` değerindedir. 999 sayısı gerçek bir gün sayısı değil, "müşterinin daha önce hiç aranmadığını" gösteren özel bir koddur. Bunu sayısal bir büyüklük olarak bırakmak özellikle Lojistik Regresyon, KNN ve SVM gibi ölçek/mesafe duyarlı modelleri saptırır. Bu nedenle ortak `veri_yukle()` fonksiyonunda `daha_once_aranmis` (0/1) ikili değişkeni türetilmiş ve `pdays = 999` değeri `-1` olarak yeniden kodlanmıştır.
2. **`duration` (Görüşme Süresi):** 0 ile 4.918 saniye arasında değişmektedir. Ancak bu değişken çağrı tamamlandıktan sonra bilindiği için doğrudan bir **Veri Sızıntısı (Data Leakage)** kaynağıdır ve ortak `veriyi_bol()` aşamasında model girdisinden çıkarılmaktadır.
3. **`age` ve `campaign`:** Yaşta 70+ değerler ve arama sayısında yüksek uç değerler görünmektedir. Emekli bireyler vadeli mevduat için gerçek ve değerli bir kitle olduğundan bu satırlar silinmez; sayısal ölçekleyici (`StandardScaler`) ve medyan doldurucu (`SimpleImputer(strategy='median')`) ile modellenir.

## D2. Hedef Değişkenin Kodlanması, duration Sütununun Çıkarılması ve Ortak Veri Bölme
> **Açıklama:** Hedef sütun `y` (0/1) ve `duration` sütununun çıkarılması ile %80-%20 tabakalı (stratified) veri ayrımı `src.ortak.veriyi_bol()` fonksiyonuyla tek seferde ve tüm modeller için aynı şekilde yapılır.

In [5]:
# Ortak veriyi_bol() fonksiyonu 'y' ve 'duration' sütunlarını ayırır, %80-%20 tabakalı bölme yapar
X_train, X_test, y_train, y_test = veriyi_bol(df)

# Veri sızıntısı kontrolü: duration sütunu kesinlikle çıkarılmış olmalı
assert "duration" not in X_train.columns, "HATA: duration sütunu eğitim verisinde kalmış!"
assert "duration" not in X_test.columns, "HATA: duration sütunu test verisinde kalmış!"

print(f"Toplam Örnek Sayısı          : {len(df)}")
print(f"Model Girdisi Özellik Sayısı : {X_train.shape[1]} ('duration' ve 'y' çıkarıldı, 'daha_once_aranmis' eklendi)")
print(f"Eğitim Seti (X_train) Boyutu : {X_train.shape}")
print(f"Test Seti   (X_test)  Boyutu : {X_test.shape}")
print("\nHedef Değişken Sınıf Dağılımı (Tüm Veri):")
print(df["y"].value_counts())
print("\nHedef Değişken Yüzdelik Dağılımı (%):")
print((df["y"].value_counts(normalize=True) * 100).round(2))

Toplam Örnek Sayısı          : 41188
Model Girdisi Özellik Sayısı : 20 ('duration' ve 'y' çıkarıldı, 'daha_once_aranmis' eklendi)
Eğitim Seti (X_train) Boyutu : (32950, 20)
Test Seti   (X_test)  Boyutu : (8238, 20)

Hedef Değişken Sınıf Dağılımı (Tüm Veri):
y
0    36548
1     4640
Name: count, dtype: int64

Hedef Değişken Yüzdelik Dağılımı (%):
y
0    88.73
1    11.27
Name: proportion, dtype: float64


**Yorum: Veri Sızıntısı ve Sınıf Dengesizliği:**
1. `duration` sütununun düşürülmesiyle modelin çağrı süresine bakarak sahte ve gerçek dünyada uygulanamaz bir ezber yapması engellenmiştir.
2. Hedef değişken dağılımı: 36.548 adet "Hayır" (%88.73) ve 4.640 adet "Evet" (%11.27). Veri kümesi **yüksek oranda dengesizdir (imbalanced)**. Bu nedenle eğitim sırasında modelin azınlık sınıfını (%11.27) yakalayabilmesi için `class_weight='balanced'` gibi yöntemler kullanılmalı ve değerlendirmede yalnızca `Accuracy`'ye değil, `Recall`, `Precision`, `F1-Score` ve `ROC-AUC` metriklerine odaklanılmalıdır.

## D3. Sayısal ve Kategorik Sütunların Ayrılması
> **Açıklama:** Sütunlar veri türlerine ve uygulanacak dönüşümlere göre listelere ayrılır. Hangi modellerin ölçeklendirmeye ihtiyaç duyduğu açıklanır.

In [6]:
# Özellik grupları src/preprocessing.py modülünden ortak olarak alınır (ikinci kez tanımlanmaz)
kategorik_ozellikler = mode_features + unknown_features + other_cat_features

print(f"Sayısal Özellik Sayısı ({len(numeric_features)}):", numeric_features)
print(f"\nMode ile Doldurulacak Kategorikler ({len(mode_features)}):", mode_features)
print(f"Unknown Korunacak Kategorikler ({len(unknown_features)}):", unknown_features)
print(f"Diğer Kategorik Özellikler ({len(other_cat_features)}):", other_cat_features)
print(f"\nToplam Kategorik Özellik Sayısı ({len(kategorik_ozellikler)}):", kategorik_ozellikler)

# Doğrulama: Ortak listedeki tüm sütunlar X_train sütunlarıyla birebir eşleşiyor mu?
assert set(numeric_features + kategorik_ozellikler) == set(X_train.columns), "Eksik ya da fazla sütun var!"
print("\nTüm sütunlar src/preprocessing.py tanımlarıyla birebir eşleştirildi ve doğrulandı.")

Sayısal Özellik Sayısı (10): ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'daha_once_aranmis']

Mode ile Doldurulacak Kategorikler (5): ['job', 'marital', 'education', 'housing', 'loan']
Unknown Korunacak Kategorikler (1): ['default']
Diğer Kategorik Özellikler (4): ['contact', 'month', 'day_of_week', 'poutcome']

Toplam Kategorik Özellik Sayısı (10): ['job', 'marital', 'education', 'housing', 'loan', 'default', 'contact', 'month', 'day_of_week', 'poutcome']

Tüm sütunlar src/preprocessing.py tanımlarıyla birebir eşleştirildi ve doğrulandı.


**Yorum: Hangi Modeller Ölçeklendirmeye İhtiyaç Duyar?:**
- **StandardScaler ($z = \frac{x - \mu}{\sigma}$):** Sayısal sütunların ortalamasını 0, standart sapmasını 1 yapar.
- **Lojistik Regresyon (BİZİM MODELİMİZ):** Gradyan temelli optimizasyon ve katsayı düzenlileştirmesi ($L_1 / L_2$) kullandığından, farklı ölçeklerdeki değişkenler (örneğin yaş: 17-98, nr.employed: 4963-5228) katsayı cezalandırmasını dengesizleştirir. Bu nedenle **Lojistik Regresyon için ölçeklendirme zorunludur**.
- **KNN ve SVM:** Mesafe tabanlı (Öklid mesafesi) veya marjin temelli çalıştıklarından büyük değerli değişkenler mesafeyi domine eder. Ölçeklendirme kesinlikle şarttır.
- **Ağaç Tabanlı Modeller (Random Forest, XGBoost):** Karar sınırlarını sıralı eşik değerleri (threshold) ile belirlediklerinden ölçeklendirmeye ihtiyaç duymazlar ancak ölçeklendirilmiş veriden de olumsuz etkilenmezler.
- **OneHotEncoder:** Kategorik değişkenlerde yapay bir sıralama (örneğin evli > bekar gibi) oluşturmamak için nominal kategoriler ikili (dummy) sütunlara dönüştürülür.

## D6. Ortak ColumnTransformer ve Ön İşleme Mimarisi
> **Açıklama:** Ön işleme adımları bu defterde ikinci kez tanımlanmaz; tüm ekip üyelerinin kullandığı ortak `ColumnTransformer` yapısı doğrudan `src.ortak.on_isleyici()` fonksiyonundan çağrılır.

In [7]:
# Ortak ColumnTransformer nesnesini src/ortak.py üzerinden çağırıyoruz
preprocessor = on_isleyici()

print("Ortak ColumnTransformer (src.ortak.on_isleyici) başarıyla yüklendi:")
print(preprocessor)

Ortak ColumnTransformer (src.ortak.on_isleyici) başarıyla yüklendi:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'campaign', 'pdays', 'previous',
                                  'emp.var.rate', 'cons.price.idx',
                                  'cons.conf.idx', 'euribor3m', 'nr.employed',
                                  'daha_once_aranmis']),
                                ('mode',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                           

**Yorum: Ortak Pipeline ve Güvenlik:**
1. **Tek Merkezli Mimari (`src/preprocessing.py` & `src/ortak.py`):** Sayısal sütunlarda `SimpleImputer(strategy='median') + StandardScaler()`, düşük oranlı `unknown` içeren kategoriklerde `SimpleImputer(strategy='most_frequent') + OneHotEncoder()`, `default` ve diğer kategoriklerde doğrudan `OneHotEncoder()` adımları tüm 5 model için aynı bileşenden gelir.
2. **`handle_unknown='ignore'`:** Test setinde veya gerçek hayat verilerinde daha önce görülmemiş yeni bir kategori gelmesi durumunda kodun çökmesini engeller, o kategoriyi tüm sıfırlar olarak kodlar.
3. **`sparse_output=False`:** Çıktının yoğun (dense numpy array) olmasını sağlayarak Lojistik Regresyon, KNN, SVM, Random Forest ve XGBoost modellerinin tamamıyla sorunsuz çalışmasını garanti eder.

## D7. Eğitim ve Test Verisinin Ayrılması (%80 - %20 Stratified) ve Ortak Doğrulama Ayarı
> **Açıklama:** `veriyi_bol(df)` ile elde edilen %80 eğitim / %20 test ayrımının boyut ve sınıf oranları ile `ortak_cv()` ortak 5-katlı doğrulama yapılandırması doğrulanır.

In [8]:
# Ortak veriyi_bol(df) sonucu oluşan eğitim/test şekilleri ve sınıf oranları
cv = ortak_cv()

print("================ ORTAK BÖLME SONUÇLARI ================")
print(f"Eğitim Seti (X_train) : {X_train.shape}")
print(f"Test Seti   (X_test)  : {X_test.shape}")
print(f"Eğitim Hedef (y_train): {y_train.shape}")
print(f"Test Hedef   (y_test) : {y_test.shape}")

print("\nEğitim Sınıf Oranları:")
print(y_train.value_counts(normalize=True).round(4))
print("\nTest Sınıf Oranları:")
print(y_test.value_counts(normalize=True).round(4))
print("\nOrtak Çapraz Doğrulama (Validation) Ayarı:")
print(cv)

================ ORTAK BÖLME SONUÇLARI ================
Eğitim Seti (X_train) : (32950, 20)
Test Seti   (X_test)  : (8238, 20)
Eğitim Hedef (y_train): (32950,)
Test Hedef   (y_test) : (8238,)

Eğitim Sınıf Oranları:
y
0    0.8873
1    0.1127
Name: proportion, dtype: float64

Test Sınıf Oranları:
y
0    0.8874
1    0.1126
Name: proportion, dtype: float64

Ortak Çapraz Doğrulama (Validation) Ayarı:
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


**Yorum: Tabakalı (Stratified) Bölmenin ve Veri Sızıntısı Önleminin Önemi:**
- `stratify=y` parametresi sayesinde hem eğitim (%88.73 - %11.27) hem de test kümesinde (%88.74 - %11.26) sınıf oranları tam olarak korunmuştur.
- `random_state=42` sabiti ile tüm ekip modellerinin birebir aynı eğitim/test ayrımını ve aynı `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` doğrulama katmanlarını kullanması garanti altına alınmıştır.
- **VERİ SIZINTISI ÖNLEMİ:** Ön işleme nesnemiz (`preprocessor`) henüz `fit` edilmemiştir. Eğitim ve test ayrıldıktan sonra `fit_transform` yalnızca `X_train` üzerinde, `transform` ise yalnızca `X_test` üzerinde uygulanır. Modellerin 5-katlı çapraz doğrulamasında da `Pipeline` sayesinde ön işleme her katmanda yalnızca ilgili eğitim katmanında `fit` edilir; test seti doğrulama sürecine asla dahil edilmez.

## 3. Ön İşlemenin Doğrulanması ve Fit-Transform Testi
> **Açıklama:** ColumnTransformer eğitim verisi üzerinde `fit_transform`, test verisi üzerinde yalnızca `transform` edilerek mimarinin çalıştığı doğrulanır.

In [9]:
# SADECE eğitim verisiyle fit_transform uyguluyoruz!
X_train_islenmis = preprocessor.fit_transform(X_train)

# Test verisini ise SADECE transform ediyoruz! (Test verisinden istatistik öğrenilmez)
X_test_islenmis = preprocessor.transform(X_test)

# İşlenmiş verilerde hiç NaN kalmadığını doğrula
train_nan = int(np.isnan(X_train_islenmis).sum())
test_nan = int(np.isnan(X_test_islenmis).sum())
assert train_nan == 0, f"Eğitim verisinde {train_nan} adet NaN kaldı!"
assert test_nan == 0, f"Test verisinde {test_nan} adet NaN kaldı!"

# Oluşan yeni özellik isimlerini alalım
ozellik_adlari = preprocessor.get_feature_names_out()

print("================ ÖN İŞLEME TEST RAPORU ================")
print(f"Girdi Özellik Sayısı     : {X_train.shape[1]}")
print(f"İşlenmiş Özellik Sayısı  : {X_train_islenmis.shape[1]}")
print(f"X_train_islenmis Boyutu  : {X_train_islenmis.shape}")
print(f"X_test_islenmis Boyutu   : {X_test_islenmis.shape}")
print(f"Eğitim Seti Boş Değer    : {train_nan} (0 olmalı)")
print(f"Test Seti Boş Değer      : {test_nan} (0 olmalı)")
print(f"\nİlk 10 Özellik İsmi:\n", list(ozellik_adlari[:10]))
print("\nÖn işleme mimarisi veri sızıntısı olmadan %100 başarıyla doğrulandı!")

================ ÖN İŞLEME TEST RAPORU ================
Girdi Özellik Sayısı     : 20
İşlenmiş Özellik Sayısı  : 58
X_train_islenmis Boyutu  : (32950, 58)
X_test_islenmis Boyutu   : (8238, 58)
Eğitim Seti Boş Değer    : 0 (0 olmalı)
Test Seti Boş Değer      : 0 (0 olmalı)

İlk 10 Özellik İsmi:
 ['num__age', 'num__campaign', 'num__pdays', 'num__previous', 'num__emp.var.rate', 'num__cons.price.idx', 'num__cons.conf.idx', 'num__euribor3m', 'num__nr.employed', 'num__daha_once_aranmis']

Ön işleme mimarisi veri sızıntısı olmadan %100 başarıyla doğrulandı!


**Yorum: Nihai Doğrulama, Veri Sızıntısı Kontrolü ve Ekip Modelleri İçin Çıkarımlar:**
1. `duration` ve `y` çıkarıldıktan sonra kalan 19 ham özellik ile türetilen 1 özellik (`daha_once_aranmis`) olmak üzere toplam **20 girdi özelliği**, `OneHotEncoder` sonrasında **58 sayısal özelliğe** dönüşmüştür (`X_train_islenmis`: `(32950, 58)`, `X_test_islenmis`: `(8238, 58)`).
2. Ne eğitim ne de test setinde tek bir `NaN` kalmamıştır (`0` eksik değer). Sayısal özellikler medyan ile doldurulup standartlaştırılmış, kategorik özellikler ise güvenli biçimde kodlanmıştır.
3. **Veri Sızıntısı Yoktur:** `veri_yukle()` içinde yapılan dönüşümler yalnızca sabit kurallara dayanır ve veri dağılımından parametre öğrenmez. Medyan, mod, ortalama, standart sapma ve kategori kodlamaları yalnızca `X_train` üzerinde (`fit_transform`) öğrenilmiş, `X_test` üzerinde yalnızca `transform` uygulanmıştır. Çapraz doğrulama sırasında da `src/pipeline.py` (`build_full_pipeline` / `build_smote_pipeline`) sayesinde her katmanda yalnızca eğitim fold'u `fit` edilir; böylece tüm modeller veri sızıntısına karşı korunur.
4. Bu ortak ön işleme omurgası `src/ortak.py`, `src/preprocessing.py` ve `src/pipeline.py` üzerinden tüm 5 model sahibi tarafından doğrudan çağrılabilir durumdadır.